# Train the U-Net (Regime C) on a Colab GPU

Thin launcher: runs the **same scripts and configs** as local development
(`scripts/train.py`, `scripts/evaluate.py`, `scripts/robustness.py`). No model code lives here.

Protocol: 5-fold cross-validation over `test_public`, grouped by physical part, plus one final
all-data model for deployment (see `src/defect_detection/training/supervised.py`).

**Before running**
1. *Runtime → Change runtime type → T4 GPU*.
2. Upload `sheet_metal.tar` (MVTec AD 2, extracted layout `sheet_metal/...`) to Google Drive at
   `MyDrive/mvtec_ad_2/sheet_metal.tar`.

**Output:** `MyDrive/defect_detection_runs/<run_id>.zip`, a self-contained run directory to unzip
into `artifacts/runs/` locally.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_TAR = "/content/drive/MyDrive/mvtec_ad_2/sheet_metal.tar"
DATA_ROOT = "/content/data/mvtec_ad_2"
OUT_DIR = "/content/drive/MyDrive/defect_detection_runs"

In [ ]:
%cd /content
!rm -rf repo && git clone -q https://github.com/yassineerraji/Industrial-Visual-Defect-Detection-Anomaly-Segmentation.git repo
%cd /content/repo
!git log --oneline -1
!pip install -q -e .

In [ ]:
# Copy the dataset from Drive to local disk (much faster I/O than reading from Drive).
!mkdir -p {DATA_ROOT} && tar -xf {DRIVE_TAR} -C {DATA_ROOT}
!python scripts/prepare_data.py --data-root {DATA_ROOT} --category sheet_metal

In [ ]:
# 5 folds + final model, 30 epochs each. MLflow is logged later, locally, from the run directory.
!python scripts/train.py --config configs/unet.yaml --data-root {DATA_ROOT} --no-mlflow

In [ ]:
import pathlib
RUN = sorted(pathlib.Path("artifacts/runs").glob("unet_*"))[-1].name
print("run:", RUN)
!python scripts/evaluate.py --run {RUN} --data-root {DATA_ROOT} --no-mlflow
!python scripts/robustness.py --run {RUN} --data-root {DATA_ROOT} --no-mlflow

In [ ]:
import shutil, os
os.makedirs(OUT_DIR, exist_ok=True)
archive = shutil.make_archive(f"{OUT_DIR}/{RUN}", "zip", root_dir="artifacts/runs", base_dir=RUN)
print("saved", archive, f"{os.path.getsize(archive) / 1e6:.0f} MB")